# TrustGuard prediction notebook (inference only)

Loads the trained artifact(s) uploaded next to this notebook, rebuilds the same row-local features used in training, and writes one fraud probability per test row in the original row order. No training happens here.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

In [ ]:
def make_features(df):
    """Same row-local feature engineering as training."""
    df = df.copy()
    for c in ["transaction_amount", "spend_last_24h", "account_age"]:
        df[c] = np.log1p(df[c])
    return df

In [ ]:
model, features = joblib.load("model.pkl")  # uploaded next to this notebook
test_df = pd.read_csv(TEST_PATH)
X_test = make_features(test_df)[features]
preds = model.predict_proba(X_test)[:, 1]

In [ ]:
preds = np.clip(np.asarray(preds, dtype=float), 0.0, 1.0)
assert len(preds) == len(test_df) and np.isfinite(preds).all()
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")